# Análisis de Rutas Aéreas con Algoritmos de Grafos

Este notebook prepara los datos del proyecto para que luego puedan resolverse los retos en C++. Lee los archivos `airports.dat` y `routes.dat` del dataset OpenFlights, limpia registros inválidos, elimina duplicados y exporta dos archivos listos para usar: `airports_clean.dat` y `routes_clean.dat`.

En resumen: el notebook no resuelve los algoritmos, sino que deja los datos organizados para construir el grafo de vuelos de forma confiable y reproducible.

## 1. Configuración Inicial y Funciones Auxiliares

Primero se importan las librerías necesarias y se define la base para leer CSV, calcular distancias y organizar los datos antes de construir el grafo.

In [ ]:
import csv
import math
from collections import defaultdict
import heapq

## 2. Carga y Limpieza de Datos

Aquí se leen los archivos originales, se descartan filas inválidas o incompletas y se construyen las estructuras principales: el diccionario de aeropuertos y el grafo de vuelos.

In [ ]:
def cargar_datos_aeropuertos(ruta_archivo = 'airports.dat'):
    mapa_aeropuertos = {}
    with open(ruta_archivo, 'r', encoding='utf-8') as archivo:
        csv_reader = csv.reader(archivo)
        for indice, fila in enumerate(csv_reader):
            if len(fila) < 7:
                continue
            try:
                id_aeropuerto = fila[0]
                nombre = fila[1]
                ciudad = fila[2]
                pais = fila[3]
                codigo_iata = fila[4] if fila[4] != '\\N' else None
                latitud = float(fila[6])
                longitud = float(fila[7])

                if not id_aeropuerto or id_aeropuerto == '\\N':
                    continue

                mapa_aeropuertos[id_aeropuerto] = {
                    "nombre": nombre,
                    "iata": codigo_iata,
                    "lat": latitud,
                    "lon": longitud,
                    "ciudad": ciudad,
                    "pais": pais
                }
            except (ValueError, IndexError):
                continue
    print(f"Cargados {len(mapa_aeropuertos)} aeropuertos.")
    return mapa_aeropuertos

def cargar_datos_rutas(ruta_archivo = 'routes.dat', mapa_aeropuertos = None):
    if mapa_aeropuertos is None:
        mapa_aeropuertos = {}

    grafo_vuelos = defaultdict(list)
    rutas_unicas = set()
    conteo_rutas_invalidas = 0
    conteo_rutas_validas = 0

    with open(ruta_archivo, 'r', encoding='utf-8') as archivo:
        csv_reader = csv.reader(archivo)
        for indice, fila in enumerate(csv_reader):
            if len(fila) < 6:
                continue
            try:
                id_origen = fila[3]
                id_destino = fila[5]

                if not id_origen or id_origen == '\\N' or \
                   not id_destino or id_destino == '\\N':
                    conteo_rutas_invalidas += 1
                    continue

                if id_origen not in mapa_aeropuertos or \
                   id_destino not in mapa_aeropuertos:
                    conteo_rutas_invalidas += 1
                    continue

                tupla_ruta = (id_origen, id_destino)
                if tupla_ruta in rutas_unicas:
                    continue
                rutas_unicas.add(tupla_ruta)

                grafo_vuelos[id_origen].append(id_destino)
                conteo_rutas_validas += 1

            except (ValueError, IndexError):
                conteo_rutas_invalidas += 1
                continue
    print(f"Cargadas {conteo_rutas_validas} rutas. ({conteo_rutas_invalidas} rutas inválidas/ignoradas)")
    return grafo_vuelos

todos_los_aeropuertos = cargar_datos_aeropuertos()
grafo_de_vuelos = cargar_datos_rutas(mapa_aeropuertos=todos_los_aeropuertos)

Cargados 7698 aeropuertos.
Cargadas 36907 rutas. (892 rutas inválidas/ignoradas)


In [ ]:
def exportar_datos_limpios(mapa_aeropuertos, grafo_vuelos, archivo_airports_out='airports_clean.dat', archivo_routes_out='routes_clean.dat'):
    print("--- Iniciando Exportación de Datos Limpios ---")

    conteo_aeros = 0
    with open(archivo_airports_out, 'w', encoding='utf-8') as f:
        for id_a, datos in mapa_aeropuertos.items():

            nombre_limpio = datos['nombre'].replace(',', ' ')


            codigo_iata = datos['iata'] if datos['iata'] and datos['iata'] != '\\N' else "N/A"
            latitud = datos['lat']
            longitud = datos['lon']


            f.write(f"{id_a},{codigo_iata},{latitud},{longitud},{nombre_limpio}\n")
            conteo_aeros += 1

    conteo_rutas = 0
    with open(archivo_routes_out, 'w', encoding='utf-8') as f:
        for id_origen, destinos in grafo_vuelos.items():
            for id_destino in destinos:
                f.write(f"{id_origen},{id_destino}\n")
                conteo_rutas += 1

    print(f"¡Éxito!")
    print(f"-> Se exportaron {conteo_aeros} aeropuertos en '{archivo_airports_out}'")
    print(f"-> Se exportaron {conteo_rutas} rutas en '{archivo_routes_out}'")

Cargados 7698 aeropuertos.
Cargadas 36907 rutas. (892 rutas inválidas/ignoradas)


### Exportar datos limpios

Finalmente, se llama a `exportar_datos_limpios` para guardar los datos procesados en `airports_clean.dat` y `routes_clean.dat`, que son los archivos que consume el programa en C++.

In [ ]:
exportar_datos_limpios(todos_los_aeropuertos, grafo_de_vuelos)

--- Iniciando Exportación de Datos Limpios ---
¡Éxito!
-> Se exportaron 7698 aeropuertos en 'airports_clean.dat'
-> Se exportaron 36907 rutas en 'routes_clean.dat'
